# Phase 07D: Mobile Runtime & Quantization Compatibility Preflight

Phase 07D kiểm tra khả năng tạo ứng viên INT8 động từ ba Student FP32 đã vượt
parity ở Phase 07C. Đây là **preflight kỹ thuật**, chưa phải đánh giá chất lượng
caption đầy đủ và chưa phải benchmark cuối trên điện thoại.

Các nguyên tắc:

- checkpoint và seed giữ nguyên: T1–S1/123, T2–S2/2026, T3–S3/42;
- FP32 TorchScript của Phase 07C là đối chứng bắt buộc;
- T3 sử dụng encoder trace không freeze đã được Phase 07C xác nhận;
- thử hai chiến lược: `decoder_dynamic_int8_linear` và
  `encoder_decoder_dynamic_int8_linear`;
- chỉ thay representation của `nn.Linear`; không huấn luyện và không QAT;
- kiểm tra 16 ảnh từ cohort Validation đã khóa ở Phase 07C;
- latency CPU chỉ được báo cáo tham khảo, không dùng làm bằng chứng Android;
- Held-out Test luôn `SEALED`.

API `torch.ao.quantization.quantize_dynamic` chỉ được dùng như probe tương thích.
Nếu API không tồn tại hoặc một kiến trúc không export được, notebook ghi nhận
kết quả âm và giữ FP32; không cài package từ Internet và không hạ gate.

## Kaggle inputs bắt buộc

1. Phase 07C Export Parity Validation CPU Benchmark.
2. Phase 07B Portable Model Factory Export.
3. Phase 02A Shared Student Pipeline.
4. Phase 01 MS COCO Data Contract.
5. T1–S1 Sequence-KD seed 123.
6. T2–S2 Sequence-KD seed 2026.
7. T3–S3 Logit-KD seed 42.
8. Dataset `mscoco2014-1`.

## Runtime

- Accelerator: `None / CPU`.
- Internet: `Off`.
- Không mở Test.


## Cell 1 — Authorization và upstream integrity


In [1]:
import copy, csv, gc, hashlib, importlib.util, json, os, platform, shutil, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

STARTED = time.perf_counter()
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//phase7_mobile_runtime_quantization_preflight",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
CANDIDATE = OUT / "candidates"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, CANDIDATE, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

CPU_THREADS = 1
torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(1)
    INTEROP_THREADS = 1
except RuntimeError:
    INTEROP_THREADS = torch.get_num_interop_threads()
torch.set_grad_enabled(False)
torch.manual_seed(42)
np.random.seed(42)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name, root=NOTEBOOK_ROOT):
    matches = sorted(Path(root).rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target == root or root in target.parents
    return target

def verify_manifest(marker_path, marker, manifest_key="manifest"):
    reference = marker[manifest_key]["csv"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    expected = int(reference.get("rows", marker.get("manifest_rows", len(rows))))
    assert len(rows) == expected
    for row in rows:
        target = resolve_inside(marker_path.parent, row["relative_path"])
        assert target.is_file(), target
        assert target.stat().st_size == int(row["size_bytes"])
        assert sha256(target) == row["sha256"]
    return rows

P7C_PATH = exactly_one(
    "PHASE7C_EXPORT_PARITY_VALIDATION_CPU_BENCHMARK_COMPLETE.json"
)
AUTH_PATH = exactly_one(
    "PHASE7D_MOBILE_OPTIMIZATION_READINESS_AUTHORIZED.json"
)
P7B_PATH = exactly_one(
    "PHASE7B_PORTABLE_MODEL_FACTORY_EXPORT_COMPLETE.json"
)
P2_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P7C, AUTH, P7B, P2, P1 = map(
    read_json, [P7C_PATH, AUTH_PATH, P7B_PATH, P2_PATH, P1_PATH]
)

assert P7C["status"] == "complete"
assert P7C["decision"] == "fp32_real_validation_parity_passed_phase7d_authorized"
assert P7C["fp32_parity"] == "3 / 3 passed"
assert AUTH["status"] == "authorized"
assert AUTH["constraints"]["new_training"] is False
assert AUTH["constraints"]["checkpoint_reselection"] is False
assert AUTH["constraints"]["validation_only"] is True
assert AUTH["constraints"]["held_out_test_opening"] is False
assert AUTH["test_set_status"] == "sealed" and AUTH["test_set_used"] is False
assert P7B["status"] == "complete"
assert P2["status"] == "ready"
assert P1["status"] == "ready"
assert P1["test_set_status"] == "sealed" and P1["test_used"] is False

p7c_rows = verify_manifest(P7C_PATH, P7C)
p7b_rows = verify_manifest(P7B_PATH, P7B)
p2_rows = verify_manifest(P2_PATH, P2, manifest_key="manifests")
assert len(p2_rows) == int(P2["artifact_count"]) == 38

entry = {
    "schema_version": "phase7d_cell01_entry_v1",
    "created_at_utc": now(),
    "status": "passed",
    "phase7c_manifest": f"{len(p7c_rows)} / {len(p7c_rows)} verified",
    "phase7b_manifest": f"{len(p7b_rows)} / {len(p7b_rows)} verified",
    "phase2_manifest": f"{len(p2_rows)} / {len(p2_rows)} verified",
    "authorized_pairs": [row["pair"] for row in AUTH["eligible_pairs"]],
    "runtime": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "device": "cpu",
        "intraop_threads": torch.get_num_threads(),
        "interop_threads": INTEROP_THREADS,
    },
    "new_training_started": False,
    "checkpoint_reselection_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
}
assert entry["authorized_pairs"] == ["T1-S1", "T2-S2", "T3-S3"]
write_json(AUDIT / "phase7d_cell01_entry.json", entry)

print("PHASE 07D CELL 1 — ENTRY")
print("Status                  : PASSED")
print("Phase 07C manifest      :", len(p7c_rows), "/", len(p7c_rows), "VERIFIED")
print("Phase 07B manifest      :", len(p7b_rows), "/", len(p7b_rows), "VERIFIED")
print("Phase 02A manifest      :", len(p2_rows), "/", len(p2_rows), "VERIFIED")
print("Authorized pairs        : T1-S1 / T2-S2 / T3-S3")
print("New training            : NOT STARTED")
print("Held-out Test           : SEALED")
print("Progress                : 1 / 6 code cells")


PHASE 07D CELL 1 — ENTRY
Status                  : PASSED
Phase 07C manifest      : 12 / 12 VERIFIED
Phase 07B manifest      : 15 / 15 VERIFIED
Phase 02A manifest      : 38 / 38 VERIFIED
Authorized pairs        : T1-S1 / T2-S2 / T3-S3
New training            : NOT STARTED
Held-out Test           : SEALED
Progress                : 1 / 6 code cells


## Cell 2 — Runtime và quantization capability discovery


In [2]:
STARTED = time.perf_counter()
SUPPORTED_ENGINES = list(torch.backends.quantized.supported_engines)
if "qnnpack" in SUPPORTED_ENGINES:
    QUANT_ENGINE = "qnnpack"
elif "x86" in SUPPORTED_ENGINES:
    QUANT_ENGINE = "x86"
elif "fbgemm" in SUPPORTED_ENGINES:
    QUANT_ENGINE = "fbgemm"
else:
    QUANT_ENGINE = None
if QUANT_ENGINE is not None:
    torch.backends.quantized.engine = QUANT_ENGINE

HAS_TORCHAO_PACKAGE = importlib.util.find_spec("torchao") is not None
HAS_EXECUTORCH_PACKAGE = importlib.util.find_spec("executorch") is not None
LEGACY_DYNAMIC_API = getattr(
    getattr(torch, "ao", None), "quantization", None
)
LEGACY_QUANTIZE_DYNAMIC = (
    getattr(LEGACY_DYNAMIC_API, "quantize_dynamic", None)
    if LEGACY_DYNAMIC_API is not None else None
)
LEGACY_DYNAMIC_AVAILABLE = callable(LEGACY_QUANTIZE_DYNAMIC)

PRECHECK_IMAGES = 16
WARMUP_RUNS = 5
MEASURED_RUNS = 10
GATES = {
    "encoder_cosine_mean_minimum": 0.99,
    "decoder_logits_cosine_mean_minimum": 0.99,
    "decoder_top1_agreement_mean_minimum": 0.95,
    "package_size_reduction_required": True,
    "finite_outputs_required": True,
    "artifact_reload_required": True,
}
strategies = [
    {
        "name": "decoder_dynamic_int8_linear",
        "quantize_encoder_linear": False,
        "quantize_decoder_linear": True,
    },
    {
        "name": "encoder_decoder_dynamic_int8_linear",
        "quantize_encoder_linear": True,
        "quantize_decoder_linear": True,
    },
]
capability = {
    "schema_version": "phase7d_runtime_capability_v1",
    "created_at_utc": now(),
    "status": "passed",
    "torch_version": torch.__version__,
    "supported_quantized_engines": SUPPORTED_ENGINES,
    "selected_probe_engine": QUANT_ENGINE,
    "legacy_torch_ao_dynamic_api_available": LEGACY_DYNAMIC_AVAILABLE,
    "torchao_package_available": HAS_TORCHAO_PACKAGE,
    "executorch_package_available": HAS_EXECUTORCH_PACKAGE,
    "legacy_api_role": "compatibility probe only; not final runtime commitment",
    "candidate_strategies": strategies,
    "preflight_gates": GATES,
    "validation_images": PRECHECK_IMAGES,
    "new_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
}
write_json(CONFIG / "phase7d_runtime_quantization_contract.json", capability)
write_json(AUDIT / "phase7d_cell02_capability.json", capability)

print("PHASE 07D CELL 2 — CAPABILITY")
print("Status                  : PASSED")
print("PyTorch                 :", torch.__version__)
print("Quantized engines       :", SUPPORTED_ENGINES)
print("Probe engine            :", QUANT_ENGINE)
print("Legacy dynamic API      :", LEGACY_DYNAMIC_AVAILABLE)
print("torchao / ExecuTorch    :", HAS_TORCHAO_PACKAGE, "/", HAS_EXECUTORCH_PACKAGE)
print("Validation preflight    :", PRECHECK_IMAGES, "images")
print("Held-out Test           : SEALED")
print("Progress                : 2 / 6 code cells")


PHASE 07D CELL 2 — CAPABILITY
Status                  : PASSED
PyTorch                 : 2.11.0+cpu
Quantized engines       : ['qnnpack', 'onednn', 'x86', 'fbgemm']
Probe engine            : qnnpack
Legacy dynamic API      : True
torchao / ExecuTorch    : True / False
Validation preflight    : 16 images
Held-out Test           : SEALED
Progress                : 2 / 6 code cells


## Cell 3 — Source registry, cohort và strict checkpoint load


In [3]:
STARTED = time.perf_counter()
import importlib.util

from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

P7C_ROOT = P7C_PATH.parent
P7B_ROOT = P7B_PATH.parent
P2_ROOT = P2_PATH.parent

# Factory và architecture contracts.
factory_audit = read_json(P7B_ROOT / "audits/phase7b_cell02_factory.json")
FACTORY_PATH = resolve_inside(
    P7B_ROOT, factory_audit["factory_file"]["relative_path"]
)
assert sha256(FACTORY_PATH) == factory_audit["factory_file"]["sha256"]
spec = importlib.util.spec_from_file_location(
    "phase7b_portable_model_factory", FACTORY_PATH
)
factory = importlib.util.module_from_spec(spec)
spec.loader.exec_module(factory)

contract_paths = {
    "decoder": P2_ROOT / "configs/decoder/shared_transformer_decoder_contract.json",
    "S1": P2_ROOT / "configs/encoders/s1_encoder_adapter_contract.json",
    "S2": P2_ROOT / "configs/encoders/s2_encoder_adapter_contract.json",
    "S3": P2_ROOT / "configs/encoders/s3_encoder_adapter_contract.json",
}
preprocessing_paths = {
    student_id: (
        P2_ROOT
        / f"configs/preprocessing/{student_id.lower()}_preprocessing_contract.json"
    )
    for student_id in ["S1", "S2", "S3"]
}
for path in [*contract_paths.values(), *preprocessing_paths.values()]:
    assert path.is_file(), path
contracts = {key: read_json(path) for key, path in contract_paths.items()}
preprocessing_contracts = {
    key: read_json(path) for key, path in preprocessing_paths.items()
}

def build_transform(student_id):
    contract = preprocessing_contracts[student_id]
    validation = contract["shared_geometry"]["validation"]
    return transforms.Compose([
        transforms.Resize(
            int(validation["resize_shorter"]),
            interpolation=InterpolationMode.BICUBIC,
            antialias=True,
        ),
        transforms.CenterCrop(int(validation["center_crop"])),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=contract["normalization"]["mean"],
            std=contract["normalization"]["std"],
        ),
    ])

validation_transforms = {
    student_id: build_transform(student_id)
    for student_id in ["S1", "S2", "S3"]
}

# Khóa đúng 16 ảnh đầu từ cohort Phase 07C và xác minh lại hash ảnh.
cohort_csv = P7C_ROOT / "configs/phase7c_validation_image_cohort.csv"
assert cohort_csv.is_file()
with cohort_csv.open("r", encoding="utf-8", newline="") as handle:
    full_cohort = list(csv.DictReader(handle))
assert len(full_cohort) == 100
preflight_cohort = full_cohort[:PRECHECK_IMAGES]

known_val_roots = [
    INPUT_ROOT / "datasets/nguynthnhdan/mscoco2014-1/val2014",
    INPUT_ROOT / "datasets/nguynthnhdan/mscoco2014-1/val2014/val2014",
    INPUT_ROOT / "mscoco2014-1/val2014",
    INPUT_ROOT / "mscoco2014-1/val2014/val2014",
]
val_candidates = {
    path.resolve() for path in known_val_roots if path.is_dir()
}
for pattern in ["*/val2014", "*/*/val2014", "*/*/*/val2014", "*/*/*/*/val2014"]:
    for path in INPUT_ROOT.glob(pattern):
        if path.is_dir():
            val_candidates.add(path.resolve())
            if (path / "val2014").is_dir():
                val_candidates.add((path / "val2014").resolve())
valid_val_roots = [
    root for root in sorted(val_candidates)
    if all((root / row["filename"]).is_file() for row in preflight_cohort)
]
if not valid_val_roots:
    dataset_root = INPUT_ROOT / "datasets"
    search_root = dataset_root if dataset_root.is_dir() else INPUT_ROOT
    recursive = {
        path.resolve() for path in search_root.rglob("val2014") if path.is_dir()
    }
    recursive |= {
        (path / "val2014").resolve()
        for path in recursive if (path / "val2014").is_dir()
    }
    valid_val_roots = [
        root for root in sorted(recursive)
        if all((root / row["filename"]).is_file() for row in preflight_cohort)
    ]
assert len(valid_val_roots) == 1, valid_val_roots
VAL_ROOT = valid_val_roots[0]
for row in preflight_cohort:
    path = VAL_ROOT / row["filename"]
    assert path.stat().st_size == int(row["size_bytes"])
    assert sha256(path) == row["sha256"]

# Checkpoint references.
p7b_entry = read_json(P7B_ROOT / "audits/phase7b_cell01_entry.json")
checkpoint_rows = []
for row in p7b_entry["checkpoints"]:
    checkpoint = exactly_one(row["checkpoint_file"])
    assert checkpoint.stat().st_size == int(row["checkpoint_size_bytes"])
    assert sha256(checkpoint) == row["checkpoint_sha256"]
    checkpoint_rows.append({**row, "resolved_path": str(checkpoint)})

# Resolve FP32 portable controls. T3 uses the Phase 07C repair.
p7b_registry = read_json(P7B_ROOT / "results/phase7b_export_registry.json")
p7b_by_pair = {row["pair"]: row for row in p7b_registry["rows"]}
t3_repair = read_json(P7C_ROOT / "audits/phase7c_t3_encoder_export_repair.json")
assert t3_repair["status"] == "passed"
assert t3_repair["weights_changed"] is False
assert t3_repair["threshold_changed"] is False

PAIR_TO_STUDENT = {"T1-S1": "S1", "T2-S2": "S2", "T3-S3": "S3"}
fp32_rows = []
for pair in ["T1-S1", "T2-S2", "T3-S3"]:
    artifact_map = {
        item["format"]: item for item in p7b_by_pair[pair]["artifacts"]
    }
    decoder_info = artifact_map["torchscript_decoder"]
    decoder_path = resolve_inside(P7B_ROOT, decoder_info["relative_path"])
    assert sha256(decoder_path) == decoder_info["sha256"]
    if pair == "T3-S3":
        encoder_info = t3_repair["repaired_artifact"]
        encoder_path = resolve_inside(P7C_ROOT, encoder_info["relative_path"])
        encoder_variant = "phase7c_unfrozen_trace_repair"
    else:
        encoder_info = artifact_map["torchscript_encoder"]
        encoder_path = resolve_inside(P7B_ROOT, encoder_info["relative_path"])
        encoder_variant = "phase7b_frozen_trace"
    assert sha256(encoder_path) == encoder_info["sha256"]
    fp32_rows.append({
        "pair": pair,
        "student_id": PAIR_TO_STUDENT[pair],
        "encoder_path": str(encoder_path),
        "decoder_path": str(decoder_path),
        "encoder_sha256": encoder_info["sha256"],
        "decoder_sha256": decoder_info["sha256"],
        "encoder_variant": encoder_variant,
        "package_size_bytes": encoder_path.stat().st_size + decoder_path.stat().st_size,
    })

write_json(CONFIG / "phase7d_preflight_cohort.json", {
    "schema_version": "phase7d_preflight_cohort_v1",
    "created_at_utc": now(),
    "source": "first 16 rows of locked Phase 07C cohort",
    "rows": preflight_cohort,
})
write_json(RESULT / "phase7d_fp32_control_registry.json", {
    "schema_version": "phase7d_fp32_control_registry_v1",
    "created_at_utc": now(),
    "rows": fp32_rows,
})
write_json(AUDIT / "phase7d_cell03_sources.json", {
    "schema_version": "phase7d_cell03_sources_v1",
    "created_at_utc": now(),
    "status": "passed",
    "checkpoint_hashes": "3 / 3 verified",
    "fp32_controls": "3 / 3 verified",
    "validation_images": PRECHECK_IMAGES,
    "caption_text_read": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07D CELL 3 — SOURCES")
print("Status                  : PASSED")
print("Checkpoint hashes       : 3 / 3 VERIFIED")
print("FP32 controls           : 3 / 3 VERIFIED")
print("T3 encoder              : PHASE 07C REPAIR")
print("Validation images       :", PRECHECK_IMAGES)
print("Caption / Test          : NOT READ / SEALED")
print("Progress                : 3 / 6 code cells")


PHASE 07D CELL 3 — SOURCES
Status                  : PASSED
Checkpoint hashes       : 3 / 3 VERIFIED
FP32 controls           : 3 / 3 VERIFIED
T3 encoder              : PHASE 07C REPAIR
Validation images       : 16
Caption / Test          : NOT READ / SEALED
Progress                : 3 / 6 code cells


## Cell 4 — Tạo và đóng gói các ứng viên dynamic INT8


In [4]:
STARTED = time.perf_counter()
VOCABULARY_SIZE = 10080
fp32_by_pair = {row["pair"]: row for row in fp32_rows}
checkpoint_by_pair = {row["pair"]: row for row in checkpoint_rows}

def make_token_probe(image_id):
    token_ids = torch.zeros((1, 31), dtype=torch.long)
    token_ids[0, 0] = 1
    for position, prime in enumerate([17, 31, 43, 59, 71, 89, 101], start=1):
        token_ids[0, position] = (
            (int(image_id) * prime + position * 97) % (VOCABULARY_SIZE - 4)
        ) + 4
    token_mask = torch.zeros((1, 31), dtype=torch.bool)
    token_mask[:, :8] = True
    return token_ids, token_mask

def load_verified_checkpoint(row):
    path = Path(row["resolved_path"])
    assert sha256(path) == row["checkpoint_sha256"]
    try:
        return torch.load(path, map_location="cpu", weights_only=True), "weights_only"
    except Exception as error:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        return payload, f"trusted_pickle_after_sha256:{type(error).__name__}"

def count_dynamic_quantized_modules(model):
    return sum(
        1 for module in model.modules()
        if "quantized.dynamic" in module.__class__.__module__
    )

def artifact_record(path, role):
    return {
        "role": role,
        "relative_path": path.relative_to(OUT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    }

candidate_rows = []
for pair in ["T1-S1", "T2-S2", "T3-S3"]:
    student_id = PAIR_TO_STUDENT[pair]
    checkpoint = checkpoint_by_pair[pair]
    fp32 = fp32_by_pair[pair]
    transform = validation_transforms[student_id]
    sample_row = preflight_cohort[0]
    with Image.open(VAL_ROOT / sample_row["filename"]) as image_handle:
        sample_image = transform(image_handle.convert("RGB")).unsqueeze(0)
    sample_ids, sample_mask = make_token_probe(int(sample_row["image_id"]))

    eager_encoder = factory.build_encoder(
        student_id, contracts[student_id]
    ).cpu().eval()
    eager_decoder = factory.build_decoder(contracts["decoder"]).cpu().eval()
    payload, load_mode = load_verified_checkpoint(checkpoint)
    encoder_result = eager_encoder.load_state_dict(
        payload["student_encoder_state"], strict=True
    )
    decoder_result = eager_decoder.load_state_dict(
        payload["student_decoder_state"], strict=True
    )
    assert not encoder_result.missing_keys and not encoder_result.unexpected_keys
    assert not decoder_result.missing_keys and not decoder_result.unexpected_keys
    del payload
    gc.collect()
    with torch.inference_mode():
        sample_memory = eager_encoder(sample_image)

    for strategy in strategies:
        candidate_dir = CANDIDATE / pair.lower().replace("-", "_") / strategy["name"]
        candidate_dir.mkdir(parents=True, exist_ok=True)
        encoder_path = candidate_dir / "encoder.ts"
        decoder_path = candidate_dir / "decoder.ts"
        row = {
            "pair": pair,
            "student_id": student_id,
            "seed": int(checkpoint["seed"]),
            "strategy": strategy["name"],
            "quantization_api": "torch.ao.quantization.quantize_dynamic",
            "quantized_engine": QUANT_ENGINE,
            "status": "not_attempted",
            "checkpoint_load_mode": load_mode,
            "encoder_dynamic_quantized_modules": 0,
            "decoder_dynamic_quantized_modules": 0,
            "artifacts": [],
            "error": None,
        }
        if not LEGACY_DYNAMIC_AVAILABLE or QUANT_ENGINE is None:
            row["status"] = "unsupported_runtime_api"
            row["error"] = (
                "Dynamic quantization API or quantized CPU engine unavailable"
            )
            candidate_rows.append(row)
            continue
        try:
            if strategy["quantize_encoder_linear"]:
                quant_encoder = LEGACY_QUANTIZE_DYNAMIC(
                    copy.deepcopy(eager_encoder),
                    {nn.Linear},
                    dtype=torch.qint8,
                    inplace=False,
                ).eval()
                row["encoder_dynamic_quantized_modules"] = (
                    count_dynamic_quantized_modules(quant_encoder)
                )
                assert row["encoder_dynamic_quantized_modules"] > 0
                with torch.inference_mode():
                    traced_encoder = torch.jit.trace(
                        quant_encoder,
                        (sample_image,),
                        strict=True,
                        check_trace=False,
                    ).eval()
                torch.jit.save(traced_encoder, str(encoder_path))
            else:
                shutil.copy2(fp32["encoder_path"], encoder_path)

            quant_decoder = LEGACY_QUANTIZE_DYNAMIC(
                copy.deepcopy(eager_decoder),
                {nn.Linear},
                dtype=torch.qint8,
                inplace=False,
            ).eval()
            row["decoder_dynamic_quantized_modules"] = (
                count_dynamic_quantized_modules(quant_decoder)
            )
            assert row["decoder_dynamic_quantized_modules"] > 0
            with torch.inference_mode():
                traced_decoder = torch.jit.trace(
                    quant_decoder,
                    (sample_ids, sample_memory, sample_mask),
                    strict=True,
                    check_trace=False,
                ).eval()
            torch.jit.save(traced_decoder, str(decoder_path))

            # Artifact reload is part of the preflight, not assumed from save().
            loaded_encoder = torch.jit.load(
                str(encoder_path), map_location="cpu"
            ).eval()
            loaded_decoder = torch.jit.load(
                str(decoder_path), map_location="cpu"
            ).eval()
            with torch.inference_mode():
                candidate_memory = loaded_encoder(sample_image)
                candidate_logits = loaded_decoder(
                    sample_ids, candidate_memory, sample_mask
                )
            assert tuple(candidate_memory.shape) == (1, 49, 256)
            assert tuple(candidate_logits.shape) == (1, 31, 10080)
            assert torch.isfinite(candidate_memory).all()
            assert torch.isfinite(candidate_logits).all()

            row["status"] = "exported_and_reloaded"
            row["artifacts"] = [
                artifact_record(encoder_path, "encoder"),
                artifact_record(decoder_path, "decoder"),
            ]
            row["package_size_bytes"] = (
                encoder_path.stat().st_size + decoder_path.stat().st_size
            )
            row["fp32_package_size_bytes"] = int(fp32["package_size_bytes"])
            row["package_reduction_fraction"] = (
                1.0
                - row["package_size_bytes"] / row["fp32_package_size_bytes"]
            )
            del (
                loaded_encoder, quant_decoder, traced_decoder, loaded_decoder,
                candidate_memory, candidate_logits,
            )
            if strategy["quantize_encoder_linear"]:
                del quant_encoder, traced_encoder
        except Exception as error:
            row["status"] = "failed"
            row["error"] = f"{type(error).__name__}: {error}"
        candidate_rows.append(row)
        gc.collect()

    del eager_encoder, eager_decoder, sample_image, sample_ids, sample_mask, sample_memory
    gc.collect()

write_json(RESULT / "phase7d_candidate_export_registry.json", {
    "schema_version": "phase7d_candidate_export_registry_v1",
    "created_at_utc": now(),
    "rows": candidate_rows,
})
exported_count = sum(
    row["status"] == "exported_and_reloaded" for row in candidate_rows
)
write_json(AUDIT / "phase7d_cell04_candidate_export.json", {
    "schema_version": "phase7d_cell04_candidate_export_v1",
    "created_at_utc": now(),
    "status": "passed",
    "attempted": len(candidate_rows),
    "exported_and_reloaded": exported_count,
    "negative_results_are_valid_preflight_evidence": True,
    "weights_changed_by_training": False,
    "checkpoint_reselection": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07D CELL 4 — CANDIDATE EXPORT")
print("Status                  : PASSED")
for row in candidate_rows:
    print(f"{row['pair']} | {row['strategy']} : {row['status']}")
print("Exported/reloaded       :", exported_count, "/", len(candidate_rows))
print("New training            : NOT PERFORMED")
print("Held-out Test           : SEALED")
print("Progress                : 4 / 6 code cells")


/tmp/ipykernel_16/145893383.py:118: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quant_decoder = LEGACY_QUANTIZE_DYNAMIC(
[W1008 03:57:49.644235350 qlinear_dynamic.cpp:251] Warning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (function operator())
/tmp/ipykernel_16/145893383.py:97: DeprecationWarning: 

PHASE 07D CELL 4 — CANDIDATE EXPORT
Status                  : PASSED
T1-S1 | decoder_dynamic_int8_linear : exported_and_reloaded
T1-S1 | encoder_decoder_dynamic_int8_linear : exported_and_reloaded
T2-S2 | decoder_dynamic_int8_linear : exported_and_reloaded
T2-S2 | encoder_decoder_dynamic_int8_linear : exported_and_reloaded
T3-S3 | decoder_dynamic_int8_linear : exported_and_reloaded
T3-S3 | encoder_decoder_dynamic_int8_linear : exported_and_reloaded
Exported/reloaded       : 6 / 6
New training            : NOT PERFORMED
Held-out Test           : SEALED
Progress                : 4 / 6 code cells


## Cell 5 — Validation-only compatibility, size và CPU reference


In [5]:
STARTED = time.perf_counter()
import statistics

def cosine(left, right):
    return float(F.cosine_similarity(
        left.float().reshape(1, -1),
        right.float().reshape(1, -1),
        dim=1,
        eps=1e-12,
    ).item())

def percentile(values, value):
    return float(np.percentile(np.asarray(values, dtype=np.float64), value))

evaluation_rows = []
for candidate in candidate_rows:
    if candidate["status"] != "exported_and_reloaded":
        evaluation_rows.append({
            "pair": candidate["pair"],
            "student_id": candidate["student_id"],
            "seed": candidate["seed"],
            "strategy": candidate["strategy"],
            "status": "not_evaluated",
            "eligible_for_phase7e": False,
            "reason": candidate["error"] or candidate["status"],
        })
        continue

    pair = candidate["pair"]
    student_id = candidate["student_id"]
    fp32 = fp32_by_pair[pair]
    transform = validation_transforms[student_id]
    artifact_map = {item["role"]: item for item in candidate["artifacts"]}
    candidate_encoder_path = resolve_inside(
        OUT, artifact_map["encoder"]["relative_path"]
    )
    candidate_decoder_path = resolve_inside(
        OUT, artifact_map["decoder"]["relative_path"]
    )
    fp32_encoder = torch.jit.load(fp32["encoder_path"], map_location="cpu").eval()
    fp32_decoder = torch.jit.load(fp32["decoder_path"], map_location="cpu").eval()
    candidate_encoder = torch.jit.load(
        str(candidate_encoder_path), map_location="cpu"
    ).eval()
    candidate_decoder = torch.jit.load(
        str(candidate_decoder_path), map_location="cpu"
    ).eval()

    first = preflight_cohort[0]
    with Image.open(VAL_ROOT / first["filename"]) as image_handle:
        warmup_image = transform(image_handle.convert("RGB")).unsqueeze(0)
    warmup_ids, warmup_mask = make_token_probe(int(first["image_id"]))
    with torch.inference_mode():
        for _ in range(WARMUP_RUNS):
            warmup_memory = candidate_encoder(warmup_image)
            _ = candidate_decoder(warmup_ids, warmup_memory, warmup_mask)

    memory_cosines = []
    logits_cosines = []
    top1_agreements = []
    latencies = []
    for index, cohort in enumerate(preflight_cohort):
        with Image.open(VAL_ROOT / cohort["filename"]) as image_handle:
            image = transform(image_handle.convert("RGB")).unsqueeze(0)
        token_ids, token_mask = make_token_probe(int(cohort["image_id"]))
        with torch.inference_mode():
            fp32_memory = fp32_encoder(image)
            fp32_logits = fp32_decoder(token_ids, fp32_memory, token_mask)
            started_ns = time.perf_counter_ns()
            candidate_memory = candidate_encoder(image)
            candidate_logits = candidate_decoder(
                token_ids, candidate_memory, token_mask
            )
            latency_ms = (time.perf_counter_ns() - started_ns) / 1e6
        assert tuple(candidate_memory.shape) == (1, 49, 256)
        assert tuple(candidate_logits.shape) == (1, 31, 10080)
        assert torch.isfinite(candidate_memory).all()
        assert torch.isfinite(candidate_logits).all()
        memory_cosines.append(cosine(fp32_memory, candidate_memory))
        logits_cosines.append(cosine(fp32_logits, candidate_logits))
        top1_agreements.append(float(
            (
                fp32_logits.argmax(dim=-1)
                == candidate_logits.argmax(dim=-1)
            ).float().mean().item()
        ))
        if index < MEASURED_RUNS:
            latencies.append(latency_ms)

    memory_cosine_mean = statistics.mean(memory_cosines)
    logits_cosine_mean = statistics.mean(logits_cosines)
    top1_agreement_mean = statistics.mean(top1_agreements)
    package_reduction = float(candidate["package_reduction_fraction"])
    gates = {
        "encoder_cosine": (
            memory_cosine_mean >= GATES["encoder_cosine_mean_minimum"]
        ),
        "decoder_logits_cosine": (
            logits_cosine_mean
            >= GATES["decoder_logits_cosine_mean_minimum"]
        ),
        "decoder_top1_agreement": (
            top1_agreement_mean
            >= GATES["decoder_top1_agreement_mean_minimum"]
        ),
        "package_size_reduction": package_reduction > 0.0,
        "finite_and_reload": True,
    }
    eligible = all(gates.values())
    evaluation_rows.append({
        "pair": pair,
        "student_id": student_id,
        "seed": candidate["seed"],
        "strategy": candidate["strategy"],
        "status": "evaluated",
        "eligible_for_phase7e": eligible,
        "gates": gates,
        "validation_images": PRECHECK_IMAGES,
        "memory_cosine_mean": memory_cosine_mean,
        "decoder_logits_cosine_mean": logits_cosine_mean,
        "decoder_top1_agreement_mean": top1_agreement_mean,
        "decoder_top1_agreement_min": min(top1_agreements),
        "fp32_package_size_bytes": int(candidate["fp32_package_size_bytes"]),
        "candidate_package_size_bytes": int(candidate["package_size_bytes"]),
        "package_reduction_fraction": package_reduction,
        "cpu_reference_p50_ms": percentile(latencies, 50),
        "cpu_reference_p95_ms": percentile(latencies, 95),
        "quantized_engine": candidate["quantized_engine"],
        "artifacts": candidate["artifacts"],
        "reason": None if eligible else "one_or_more_compatibility_gates_failed",
    })
    del (
        fp32_encoder, fp32_decoder, candidate_encoder, candidate_decoder,
        warmup_image, warmup_ids, warmup_mask, warmup_memory,
    )
    gc.collect()

selected_rows = []
fp32_fallback_pairs = []
for pair in ["T1-S1", "T2-S2", "T3-S3"]:
    eligible = [
        row for row in evaluation_rows
        if row["pair"] == pair and row["eligible_for_phase7e"]
    ]
    if eligible:
        eligible.sort(
            key=lambda row: (
                row["candidate_package_size_bytes"],
                row["cpu_reference_p50_ms"],
                row["strategy"],
            )
        )
        selected_rows.append(eligible[0])
    else:
        fp32_fallback_pairs.append(pair)

write_json(RESULT / "phase7d_compatibility_results.json", {
    "schema_version": "phase7d_compatibility_results_v1",
    "created_at_utc": now(),
    "gates": GATES,
    "rows": evaluation_rows,
})
write_json(RESULT / "phase7d_selected_candidate_registry.json", {
    "schema_version": "phase7d_selected_candidate_registry_v1",
    "created_at_utc": now(),
    "selection_scope": "technical compatibility preflight only",
    "selection_rule": "eligible candidate with smallest package; CPU latency tie-break",
    "selected": selected_rows,
    "fp32_fallback_pairs": fp32_fallback_pairs,
})
write_json(AUDIT / "phase7d_cell05_compatibility.json", {
    "schema_version": "phase7d_cell05_compatibility_v1",
    "created_at_utc": now(),
    "status": "passed",
    "evaluated_candidates": sum(
        row["status"] == "evaluated" for row in evaluation_rows
    ),
    "eligible_candidates": sum(
        row["eligible_for_phase7e"] for row in evaluation_rows
    ),
    "selected_pairs": [row["pair"] for row in selected_rows],
    "fp32_fallback_pairs": fp32_fallback_pairs,
    "latency_used_as_final_android_evidence": False,
    "caption_quality_evaluated": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07D CELL 5 — COMPATIBILITY")
print("Status                  : PASSED")
for row in evaluation_rows:
    if row["status"] == "evaluated":
        print(
            f"{row['pair']} | {row['strategy']} | "
            f"eligible={row['eligible_for_phase7e']} | "
            f"size reduction={100*row['package_reduction_fraction']:.2f}% | "
            f"top1={row['decoder_top1_agreement_mean']:.4f}"
        )
print("Selected quantized      :", [row["pair"] for row in selected_rows])
print("FP32 fallback           :", fp32_fallback_pairs)
print("Caption quality         : NOT YET EVALUATED")
print("Held-out Test           : SEALED")
print("Progress                : 5 / 6 code cells")


PHASE 07D CELL 5 — COMPATIBILITY
Status                  : PASSED
T1-S1 | decoder_dynamic_int8_linear | eligible=True | size reduction=12.83% | top1=0.9839
T1-S1 | encoder_decoder_dynamic_int8_linear | eligible=True | size reduction=13.12% | top1=0.9879
T2-S2 | decoder_dynamic_int8_linear | eligible=True | size reduction=10.59% | top1=0.9919
T2-S2 | encoder_decoder_dynamic_int8_linear | eligible=True | size reduction=29.75% | top1=0.9859
T3-S3 | decoder_dynamic_int8_linear | eligible=True | size reduction=9.98% | top1=0.9919
T3-S3 | encoder_decoder_dynamic_int8_linear | eligible=False | size reduction=40.79% | top1=0.9355
Selected quantized      : ['T1-S1', 'T2-S2', 'T3-S3']
FP32 fallback           : []
Caption quality         : NOT YET EVALUATED
Held-out Test           : SEALED
Progress                : 5 / 6 code cells


## Cell 6 — Đóng Phase 07D và cấp quyền Phase 07E


In [6]:
STARTED = time.perf_counter()
phase7e_mode = (
    "quantized_validation_quality_gate"
    if selected_rows else "fp32_runtime_packaging"
)
authorization = {
    "schema_version": "phase7e_validation_quality_or_fp32_packaging_authorized_v1",
    "created_at_utc": now(),
    "status": "authorized",
    "mode": phase7e_mode,
    "required_next": (
        "Phase 07E full Karpathy Validation caption-quality gate for selected "
        "quantized candidates, with FP32 controls; retain FP32 for unsupported pairs"
        if selected_rows
        else "Phase 07E retain FP32 and prepare modern mobile-runtime export"
    ),
    "selected_quantized_candidates": selected_rows,
    "fp32_fallback_pairs": fp32_fallback_pairs,
    "quality_protocol": {
        "split": "Karpathy Validation",
        "images": 5000,
        "references": 25010,
        "decoding": "beam search size 3, length penalty 0.7",
        "metrics": ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D", "CE"],
        "quantized_minus_fp32_gates": {
            "CIDEr-D_minimum_delta": -0.02,
            "BLEU-4_minimum_delta": -0.01,
            "CE_maximum_delta": 0.05,
        },
        "candidate_selection_by_test_forbidden": True,
    },
    "runtime_policy": {
        "legacy_torch_ao_result_is_compatibility_evidence_only": True,
        "modern_android_runtime_export_still_required": True,
        "physical_android_benchmark_required": True,
    },
    "new_training": False,
    "checkpoint_reselection": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(
    OUT / "PHASE7E_VALIDATION_QUALITY_OR_FP32_PACKAGING_AUTHORIZED.json",
    authorization,
)

completion = {
    "schema_version": "phase7d_mobile_runtime_quantization_preflight_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "decision": (
        "compatibility_preflight_complete_phase7e_quantized_quality_authorized"
        if selected_rows
        else "compatibility_preflight_complete_fp32_retained"
    ),
    "attempted_candidates": len(candidate_rows),
    "exported_candidates": sum(
        row["status"] == "exported_and_reloaded" for row in candidate_rows
    ),
    "eligible_candidates": sum(
        row["eligible_for_phase7e"] for row in evaluation_rows
    ),
    "selected_quantized_pairs": [row["pair"] for row in selected_rows],
    "fp32_fallback_pairs": fp32_fallback_pairs,
    "caption_quality_evaluated": False,
    "modern_android_package_created": False,
    "physical_android_benchmark_performed": False,
    "new_training_performed": False,
    "checkpoint_reselection_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}

manifest_rows = []
for directory in [AUDIT, CONFIG, RESULT, CANDIDATE]:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            manifest_rows.append({
                "relative_path": path.relative_to(OUT).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256(path),
            })
manifest_csv = MANIFEST / "phase7d_manifest.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = MANIFEST / "phase7d_manifest.json"
write_json(manifest_json, {
    "schema_version": "phase7d_manifest_v1",
    "created_at_utc": now(),
    "rows": manifest_rows,
})
sha_file = MANIFEST / "SHA256SUMS"
sha_file.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in manifest_rows
    ),
    encoding="utf-8",
)
completion["manifest"] = {
    "csv": {
        "relative_path": manifest_csv.relative_to(OUT).as_posix(),
        "sha256": sha256(manifest_csv),
        "rows": len(manifest_rows),
    },
    "json": {
        "relative_path": manifest_json.relative_to(OUT).as_posix(),
        "sha256": sha256(manifest_json),
    },
    "sha256sums": {
        "relative_path": sha_file.relative_to(OUT).as_posix(),
        "sha256": sha256(sha_file),
    },
}
completion["manifest_rows"] = len(manifest_rows)
completion["elapsed_seconds_cell06"] = time.perf_counter() - STARTED
write_json(
    OUT / "PHASE7D_MOBILE_RUNTIME_QUANTIZATION_PREFLIGHT_COMPLETE.json",
    completion,
)

print("PHASE 07D — FINAL")
print("Status                  : COMPLETE")
print("Attempted candidates    :", len(candidate_rows))
print("Exported candidates     :", completion["exported_candidates"])
print("Eligible candidates     :", completion["eligible_candidates"])
print("Selected quantized      :", completion["selected_quantized_pairs"])
print("FP32 fallback           :", fp32_fallback_pairs)
print("Caption quality         : NOT YET")
print("Android final benchmark : NOT YET")
print("Held-out Test           : SEALED / NOT USED")
print("Phase 07E mode          :", phase7e_mode)
print("Manifest                :", len(manifest_rows), "/", len(manifest_rows), "VERIFIED")
print("Progress                : 6 / 6 code cells")


PHASE 07D — FINAL
Status                  : COMPLETE
Attempted candidates    : 6
Exported candidates     : 6
Eligible candidates     : 5
Selected quantized      : ['T1-S1', 'T2-S2', 'T3-S3']
FP32 fallback           : []
Caption quality         : NOT YET
Android final benchmark : NOT YET
Held-out Test           : SEALED / NOT USED
Phase 07E mode          : quantized_validation_quality_gate
Manifest                : 23 / 23 VERIFIED
Progress                : 6 / 6 code cells
